# Project 3


# Gendered Mobility in Citibike Usage

Citibike plays a central role in New York City’s transportation network, and its detailed trip records allow for a close examination of how people move through urban space. Because each ride includes the rider’s self-reported gender and precise trip timing, Citibike's data offers a rare opportunity to examine gendered mobility across different times of day and context.

The central research question is: 
**Do women use Citibike disproportionately less than men at night compared to daytime hours?**

Citibike’s data format changed in mid-2021 and no longer includes rider gender, making recent years unsuitable for gender-based analysis. The year **2019** is therefore the most recent complete year with reliable gender information. It also reflects typical pre-pandemic mobility patterns and provides a consistent schema across all twelve monthly files. With high trip volumes and clean demographic fields, 2019 offers a strong foundation for analyzing how gender differences in bikeshare usage vary between daytime and nighttime hours.

The following section outlines the specific datasets, variables, and analytical choices used in this project.

## *Project Specification*

- **Dataset to be used:**  
  Citibike Trip Histories (2019): https://citibikenyc.com/system-data

- **Analysis question:**  
  Did women use Citibike disproportionately less than men during nighttime hours (8pm–5:59am) compared to daytime hours (6am–7:59pm) in 2019?

- **Columns that will likely be used:**
  - "starttime": to extract hour of day
  - "gender": rider gender (0 = unknown, 1 = male, 2 = female)

  **Derived columns:**
  - "hour": ride start hour
  - "gender_label": readable gender categories
  - "is_night": indicator for nighttime vs. daytime

- **Hypothesis:**  
  Women’s share of Citibike rides will decrease disproportionately during nighttime hours relative to daytime hours, reflecting heightened safety concerns and reduced comfort in traveling through public space after dark.

## *Assumptions*

This analysis relies on several key assumptions about the Citibike dataset and the interpretation of rider behavior:

1. **Self-reported gender is accurate and consistently recorded.**
The analysis assumes that riders report their gender reliably, and that Citibike consistently collects this field across all trips in 2019.

2. **Trip start time reflects the primary time-of-day decision.**
The classification of rides as “daytime” or “nighttime” assumes that the start of the trip is the relevant indicator of when a rider chose to travel.

3. **Differences in ridership reflect behavioral choice, not system availability.**
The analysis assumes that bikes and stations are equally available to riders of all genders during both day and night, and that any observed differences reflect rider behavior rather than systemic access issues.

4. **The 2019 dataset represents typical pre-pandemic usage patterns.**
Selecting 2019 assumes that mobility was not affected by major disruptions (e.g., COVID-19) and that results are not confounded by unusual travel patterns.

5. **Unknown-gender riders (coded as 0) can be excluded without biasing results.**
Since the analysis focuses on comparing male and female riders, trips with unknown gender are removed under the assumption that this exclusion does not systematically distort day–night gender patterns.

6. **Nighttime hours (8 PM–5:59 AM) reflect periods of elevated safety concerns.**
The definition of “nighttime” assumes that these hours correspond to reduced visibility, greater perceived risk, and different usage motivations compared to daytime.

### Step 1: Loading the Monthly Trip Data

In [2]:
import pandas as pd

# Some months in the CitiBike Trip Histories dataset are split into multiple CSV files
# because the number of trips is too large for a single export.
# When this happens, each file contains a portion of the month’s rides,
# so we need to load all files for that month and concatenate them
# to reconstruct the full monthly dataset.

# January 2019
jan_trips = pd.read_csv("201901-citibike-tripdata_1.csv")

# February 2019
feb_trips = pd.read_csv("201902-citibike-tripdata_1.csv")

# March 2019 (two CSV files → concatenate)
mar_1 = pd.read_csv("201903-citibike-tripdata_1.csv")
mar_2 = pd.read_csv("201903-citibike-tripdata_2.csv")
mar_trips = pd.concat([mar_1, mar_2], ignore_index=True)

# April 2019 (two CSV files → concatenate)
apr_1 = pd.read_csv("201904-citibike-tripdata_1.csv")
apr_2 = pd.read_csv("201904-citibike-tripdata_2.csv")
apr_trips = pd.concat([apr_1, apr_2], ignore_index=True)

# May 2019 (two CSV files → concatenate)
may_1 = pd.read_csv("201905-citibike-tripdata_1.csv")
may_2 = pd.read_csv("201905-citibike-tripdata_2.csv")
may_trips = pd.concat([may_1, may_2], ignore_index=True)

# June 2019 (two CSV files → concatenate)
jun_1 = pd.read_csv("201906-citibike-tripdata_1.csv")
jun_2 = pd.read_csv("201906-citibike-tripdata_2.csv")
jun_trips = pd.concat([jun_1, jun_2], ignore_index=True)

# July 2019 (three CSV files → concatenate)
jul_1 = pd.read_csv("201907-citibike-tripdata_1.csv")
jul_2 = pd.read_csv("201907-citibike-tripdata_2.csv")
jul_3 = pd.read_csv("201907-citibike-tripdata_3.csv")
jul_trips = pd.concat([jul_1, jul_2, jul_3], ignore_index=True)

# August 2019 (two CSV files → concatenate)
aug_1 = pd.read_csv("201908-citibike-tripdata_1.csv")
aug_2 = pd.read_csv("201908-citibike-tripdata_2.csv")
aug_trips = pd.concat([aug_1, aug_2], ignore_index=True)

# September 2019 (three CSV files → concatenate)
sep_1 = pd.read_csv("201909-citibike-tripdata_1.csv")
sep_2 = pd.read_csv("201909-citibike-tripdata_2.csv")
sep_3 = pd.read_csv("201909-citibike-tripdata_3.csv")
sep_trips = pd.concat([sep_1, sep_2, sep_3], ignore_index=True)

# October 2019 (three CSV files → concatenate)
oct_1 = pd.read_csv("201910-citibike-tripdata_1.csv")
oct_2 = pd.read_csv("201910-citibike-tripdata_2.csv")
oct_3 = pd.read_csv("201910-citibike-tripdata_3.csv")
oct_trips = pd.concat([oct_1, oct_2, oct_3], ignore_index=True)

# November 2019 (two CSV files → concatenate)
nov_1 = pd.read_csv("201911-citibike-tripdata_1.csv")
nov_2 = pd.read_csv("201911-citibike-tripdata_2.csv")
nov_trips = pd.concat([nov_1, nov_2], ignore_index=True)

# December 2019
dec_trips = pd.read_csv("201912-citibike-tripdata_1.csv")

# Quick check
jan_trips.shape, jul_trips.shape, dec_trips.shape


((967287, 15), (2181064, 15), (955210, 15))

### Step 2: Combine all months into one DataFrame

In [3]:
# Combine all monthly DataFrames into one large DataFrame called 'trips_2019'
# ignore_index=True gives the merged DataFrame a clean set of row numbers
trips = pd.concat(
    [
        jan_trips,
        feb_trips,
        mar_trips,
        apr_trips,
        may_trips,
        jun_trips,
        jul_trips,
        aug_trips,
        sep_trips,
        oct_trips,
        nov_trips,
        dec_trips,
    ],
    ignore_index=True,
)

# Quick check on size and structure
trips.shape


(20082103, 15)

### Step 3: Clean and standardize column names

In [4]:
# Make all column names easier to work with:
# - remove leading/trailing spaces
# - convert to lowercase
# - replace spaces with underscores
trips.columns = trips.columns.str.strip().str.lower().str.replace(" ", "_")

# Check the column names after cleaning
trips.columns


Index(['tripduration', 'starttime', 'stoptime', 'start_station_id',
       'start_station_name', 'start_station_latitude',
       'start_station_longitude', 'end_station_id', 'end_station_name',
       'end_station_latitude', 'end_station_longitude', 'bikeid', 'usertype',
       'birth_year', 'gender'],
      dtype='object')

### Step 4: Inspect the structure of the data

In [5]:
# .info() shows the number of rows, columns, and data types
trips.info()

# Look at a small sample of key columns to confirm they exist and look reasonable
trips[["starttime", "stoptime", "gender"]].head()

# Look at gender value counts to see how many unknowns there are
trips["gender"].value_counts(dropna=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20082103 entries, 0 to 20082102
Data columns (total 15 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tripduration             int64  
 1   starttime                object 
 2   stoptime                 object 
 3   start_station_id         float64
 4   start_station_name       object 
 5   start_station_latitude   float64
 6   start_station_longitude  float64
 7   end_station_id           float64
 8   end_station_name         object 
 9   end_station_latitude     float64
 10  end_station_longitude    float64
 11  bikeid                   int64  
 12  usertype                 object 
 13  birth_year               int64  
 14  gender                   int64  
dtypes: float64(6), int64(4), object(5)
memory usage: 2.2+ GB


gender
1    13751156
2     4820277
0     1510670
Name: count, dtype: int64

### Step 5: Convert starttime to datetime and create hour

In [12]:
# Convert the starttime column from text (string) to a proper datetime type
trips["starttime"] = pd.to_datetime(trips["starttime"])

# Create a new column for the hour of day (0–23) extracted from the start time
trips["hour"] = trips["starttime"].dt.hour

# Check that the new hour column looks correct
trips[["starttime", "hour"]].sample(20)


,starttime,hour
13522377,2019-09-06 22:18:44.554,22
5340322,2019-05-07 05:47:00.901,5
17241346,2019-10-25 09:14:05.764,9
17410259,2019-10-28 08:19:41.332,8
3725885,2019-04-09 07:57:37.315,7
11727221,2019-08-09 07:26:33.641,7
6097769,2019-05-19 14:39:08.029,14
174623,2019-01-06 13:02:59.416,13
5509426,2019-05-09 10:32:03.998,10
15929192,2019-10-06 11:45:54.586,11


### Step 6: Keep only riders with known gender and add labels

In [7]:
### Step 6: Keep only riders with known gender and add labels

# The gender column uses:
# 0 = unknown, 1 = male, 2 = female

# Keep only rows where gender is 1 (male) or 2 (female)
trips = trips[trips["gender"].isin([1, 2])].copy()

# Map numeric codes to readable labels
gender_map = {1: "Male", 2: "Female"}
trips["gender_label"] = trips["gender"].map(gender_map)

# Check that the mapping worked
trips[["gender", "gender_label"]].head(20)


,gender,gender_label
0,1,Male
1,1,Male
2,1,Male
3,1,Male
4,1,Male
5,2,Female
6,1,Male
7,1,Male
8,2,Female
9,1,Male


### Step 7: Create a nighttime indicator

In [11]:
# We define the hours that count as "night" as 8pm–11pm (20–23) and midnight–5am (0–5)
night_hours = list(range(20, 24)) + list(range(0, 6))

# Create a boolean column that is True if the ride started at night, False otherwise
trips["is_night"] = trips["hour"].isin(night_hours)

# Look at a few rows to confirm hour + is_night make sense
trips[["starttime", "hour", "is_night", "gender_label"]].sample(20)

,starttime,hour,is_night,gender_label
587669,2019-01-17 21:41:39.104,21,True,Male
4588888,2019-04-23 18:15:43.199,18,False,Male
7076120,2019-06-03 06:42:08.704,6,False,Male
20050455,2019-12-30 23:34:19.273,23,True,Female
10385918,2019-07-22 09:15:07.188,9,False,Male
9046810,2019-07-02 15:19:19.858,15,False,Male
12600575,2019-08-20 14:23:42.292,14,False,Male
19838134,2019-12-22 10:10:30.386,10,False,Male
14490679,2019-09-18 18:04:45.058,18,False,Male
7036116,2019-06-02 13:58:14.552,13,False,Male


### Step 8: Summarize Gender Patterns (Overall, Day vs Night, Hourly)

In [22]:
# Overall gender proportions
overall_props = trips["gender_label"].value_counts(normalize=True)
print("Overall Gender Proportions:")
display(overall_props)

print("\n" + "-" * 60 + "\n")

# Gender proportions during daytime vs nighttime
day_night_props = (
    trips.groupby(["is_night", "gender_label"])
    .size()
    .groupby(level=0)
    .apply(lambda x: x / x.sum())
)

print("Gender Proportions: Daytime vs Nighttime:")
display(day_night_props)

print("\n" + "-" * 60 + "\n")

# Hourly gender proportions across the 24-hour day
hourly_props = (
    trips.groupby(["hour", "gender_label"])
    .size()
    .groupby(level=0)
    .apply(lambda x: x / x.sum())
)

print("Hourly Gender Proportions (hours 0-4 shown):")
display(hourly_props.head(10))


Overall Gender Proportions:


gender_label
Male      0.740447
Female    0.259553
Name: proportion, dtype: float64


------------------------------------------------------------

Gender Proportions: Daytime vs Nighttime:


is_night  is_night  gender_label
False     False     Female          0.265246
                    Male            0.734754
True      True      Female          0.224797
                    Male            0.775203
dtype: float64


------------------------------------------------------------

Hourly Gender Proportions (hours 0-4 shown):


hour  hour  gender_label
0     0     Female          0.190501
            Male            0.809499
1     1     Female          0.167540
            Male            0.832460
2     2     Female          0.149058
            Male            0.850942
3     3     Female          0.124440
            Male            0.875560
4     4     Female          0.153575
            Male            0.846425
dtype: float64

### Step 9: Visualize Gender Patterns (Day vs Night, Hourly)

In [24]:
import plotly.express as px
from IPython.display import HTML

# A. Create tidy DataFrame for Day vs Night gender proportions

# Count number of rides for each combination of (is_night, gender_label)
day_night_df = (
    trips.groupby(["is_night", "gender_label"]).size().reset_index(name="count")
)

# Compute proportions within each time period (day vs night)
day_night_df["proportion"] = day_night_df.groupby("is_night")["count"].transform(
    lambda x: x / x.sum()
)

# Add a readable label for plotting
day_night_df["time_period"] = day_night_df["is_night"].map(
    {False: "Daytime", True: "Nighttime"}
)

# B. Create tidy DataFrame for hourly gender proportions

# Count number of rides for each combination of (hour, gender_label)
hourly_df = trips.groupby(["hour", "gender_label"]).size().reset_index(name="count")

# Compute gender proportions within each hour (0–23)
hourly_df["proportion"] = hourly_df.groupby("hour")["count"].transform(
    lambda x: x / x.sum()
)

# C. Plot: Day vs Night gender proportions (bar chart)

fig1 = px.bar(
    day_night_df,
    x="time_period",
    y="proportion",
    color="gender_label",
    barmode="group",
    title="Gender Proportions: Daytime vs Nighttime",
    labels={
        "time_period": "Time of Day",
        "proportion": "Proportion of Trips",
        "gender_label": "Gender",
    },
)

HTML(fig1.to_html(include_plotlyjs="cdn", full_html=False))


# -----------------------------------------------------------
# D. Plot: Hourly gender proportions across the 24-hour day (line chart)
# -----------------------------------------------------------
fig2 = px.line(
    hourly_df,
    x="hour",
    y="proportion",
    color="gender_label",
    markers=True,
    title="Gender Proportions by Hour of Day",
    labels={
        "hour": "Hour of Day (0–23)",
        "proportion": "Proportion of Trips",
        "gender_label": "Gender",
    },
)

# Show all 24 hours on the x-axis for readability
fig2.update_xaxes(dtick=1)

HTML(fig2.to_html(include_plotlyjs="cdn", full_html=False))


### Step 10: Day–Night Comparison Table

In [20]:
# Create a tidy summary table using groupby
summary_table = (
    day_night_df.groupby(["gender_label", "time_period"])["proportion"]
    .mean()
    .reset_index()
)

print("Proportion of Male and Female Riders: Day vs Night")
display(summary_table)


Proportion of Male and Female Riders: Day vs Night


,gender_label,time_period,proportion
0,Female,Daytime,0.265246
1,Female,Nighttime,0.224797
2,Male,Daytime,0.734754
3,Male,Nighttime,0.775203


## *Interpretation of Results*
The visualizations reveal clear and consistent gender differences in Citibike usage across the 24-hour day, supporting the hypothesis that women are disproportionately underrepresented at night compared to daytime hours.

#### **Day vs. Night Patterns**

The bar chart comparing daytime and nighttime proportions shows that:

- During daytime hours (6:00 AM–7:59 PM), female riders account for approximately 26.5% of all trips.

- During nighttime hours (8:00 PM–5:59 AM), the female share drops to around 22.5%, while male ridership increases correspondingly.

This shift indicates that women’s participation in bikeshare mobility decreases after dark, even though overall ridership remains high. The pattern aligns with broader research suggesting that nighttime travel may be influenced by perceived safety, lighting conditions, and environmental comfort.

#### **Hourly Patterns Across the Day**

The hourly line chart further illustrates how the gender gap varies over the full 24-hour cycle:

- Female ridership is lowest overnight, dropping to 12–15% between midnight and 5 AM.

- Female representation increases steadily in the early morning and stabilizes during daytime hours.

- The most balanced gender distribution occurs during the late morning and afternoon.

- Male ridership remains dominant throughout the day but is especially high during late-night and early-morning hours.

These hourly trends reinforce the idea that nighttime mobility presents additional barriers or concerns for women. Even within a dense and well-lit city such as New York, the gender gap in bikeshare usage widens precisely during hours typically associated with reduced visibility, lower pedestrian activity, and heightened safety concerns.

Across both visualizations, the results are consistent: **women engage in Citibike trips at much lower proportions during nighttime hours**.
This pattern suggests that the time of day plays a significant role in shaping gendered mobility, with potential implications for transportation planning, nighttime safety measures, and the equitable design of cycling infrastructure.

## *Limitations*

While this analysis provides meaningful insight into gender differences in Citibike usage across time of day, several limitations should be acknowledged:

1. Gender is self-reported and binary.
Citibike’s dataset only includes three gender codes (0 = unknown, 1 = male, 2 = female).
This excludes nonbinary identities and relies on riders’ self-reporting, which may not reflect gender diversity or full accuracy.

2. Unknown-gender riders are excluded.
Trips coded with gender = 0 were removed from the analysis.
If unknown-gender riders travel differently at night, this exclusion could subtly bias results.

3. Time-of-day categories are simplified.
The division between “daytime” (6am–7:59pm) and “nighttime” (8pm–5:59am) is somewhat arbitrary.
Small shifts in the cutoff hours could change the exact proportions, though not the general trend.

4. The analysis does not consider distance or trip purpose.
Tripduration, station locations, and route choices are not analyzed.
These could reveal additional patterns in how men and women navigate the city at different times.

5. Spatial variation is not examined.
Neighborhood-level differences may exist, but this analysis treats all rides as equally comparable regardless of where they begin.

6. Potential confounders are not included.
Factors such as weather, daylight hours, seasonality, station availability, or special events could influence usage differently across genders but were not incorporated into this project.

7. Findings apply only to 2019.
Post-pandemic mobility (after 2020) likely differs, and Citibike removed gender data after mid-2021.
The results therefore represent pre-pandemic behavior and cannot be generalized to recent years.

## Conclusion

This analysis set out to test whether women use Citibike disproportionately less than men during nighttime hours compared to daytime. The results strongly support this hypothesis. While women account for roughly 26% of all daytime trips, their share drops to about 22% at night, indicating a meaningful decline in participation after dark. Hourly patterns further reinforce this finding: the proportion of female riders steadily decreases as the evening progresses and remains lowest in the late-night and early-morning hours.

Taken together, these patterns suggest that gender disparities in Citibike usage become more pronounced at night. This likely reflects broader factors that shape women’s mobility after dark, including safety concerns, comfort levels, and the navigability of public space. The evidence from 2019 trip data therefore supports the conclusion that women are underrepresented in nighttime bikeshare usage relative to men.